# Exercise 1: image → JSON → DataFrame

## Tasks

1. **One image:** send a single picture to the OpenAI API and read its description.
2. **Structure:** ask for a fixed JSON format with a few fields.
3. **More images:** repeat for a small folder and collect the answers in a table.
4. **Evaluate:** compare the table with hand labels, field by field.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret.

This notebook uses the **OpenAI API** (model `gpt-6-luna`). Other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code. The API calls are in `helper.py` – open it to see how they work.

In [ ]:
# Standard Python libraries.
import sys
from typing import Literal

# Libraries installed from requirements.txt.
import jiwer
import pandas as pd
from pydantic import BaseModel
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# helper.py is in the main folder, one level up.
sys.path.append("..")

In [ ]:
# Functions from helper.py.
from helper import DATA, ask, describe, show_images

## 1. Send one image to the OpenAI API and read its description

Pick one image. Other images in `data/images/`: `street.jpg`, `sheep.jpg`, `parking.jpg`, `lake_night.jpg`, `lake_swan.jpg`.

In [ ]:
# The image to describe. Change the filename to try another one.
image_path = DATA / "images" / "lake.jpg"
show_images([image_path])

In [ ]:
# Send a question and the image to the OpenAI API.
answer = describe("Describe this image in two sentences.", image_path)
print(answer)

The model reads the image as tokens, so every image costs money. Estimate before you run: tokens per image × number of images × price.

In [ ]:
# Example: a folder with 1,000 images.
images = 1000
tokens_per_image = 1500  # assumed
price = 0.10  # US$ per 1 million input tokens

# Estimated cost for the whole folder.
cost = images * tokens_per_image * price / 1_000_000
print("Estimated cost: US$", cost)

## 2. Ask for a fixed JSON format

A free text is hard to compare. Now the answer must contain exactly these fields, written as a Python class.

- `text_in_image` is `None` when no text is visible.
- `inferred` holds what the model only **concludes** but cannot see, e.g. the time of day. All other fields hold what is **visible**.

In [ ]:
# The answer format: one line per field.
class ImageRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    objects: list[str]
    people_count: int
    text_in_image: str | None
    inferred: list[str]

In [ ]:
# The instruction for the model.
PROMPT = """
Describe this image.
Pick the scene from the list.
Name up to five prominent objects as lowercase singular nouns.
Count the visible people.
Copy the most prominent text. Use null if there is no text.
Use [unreadable] if the text cannot be read.
Under inferred, list up to three things you conclude but cannot see,
for example the time of day or the season.
Write unknown if the image cannot tell.
"""

In [ ]:
# Same image, now with a fixed answer format.
answer = ask(PROMPT, ImageRecord, [image_path])
answer

## 3. Repeat for all images and collect the answers in a table

In [ ]:
# All JPG images in the course folder.
folder = DATA / "images"
paths = sorted(folder.glob("*.jpg"))
print(len(paths), "images")

In [ ]:
# Ask the model about every image.
rows = []
for path in paths:
    row = ask(PROMPT, ImageRecord, [path])
    row["file"] = path.name  # we add the filename ourselves
    rows.append(row)

# One row per image.
extracted = pd.DataFrame(rows)
extracted

In [ ]:
# Save the table as JSON lines: one record per line.
extracted.to_json("extracted.jsonl", orient="records", lines=True)

## 4. Compare the table with the hand labels, field by field

The average hides the weak field – so each field gets its own score.

In [ ]:
# Hand labels: one row per image.
labels = pd.read_csv(folder / "labels.csv")

# Put model answer (_pred) and label (_ref) side by side.
scores = extracted.merge(labels, on="file", suffixes=("_pred", "_ref"))
scores

**Scene and people count:** the answer is either right or wrong. Accuracy is the share of images where model and label agree.

In [ ]:
# Accuracy for the scene.
scene = accuracy_score(scores["scene_ref"], scores["scene_pred"])
print("Scene accuracy:", round(scene, 2))

# Accuracy for the number of people.
ref = scores["people_count_ref"]
pred = scores["people_count_pred"]
people = accuracy_score(ref, pred)
print("People count accuracy:", round(people, 2))

**Objects** are a list, so "right or wrong" is too strict. The F1 score is high only if the model names the right objects (precision) **and** finds all of them (recall).

In [ ]:
# Object lists from the model and from the labels.
pred = scores["objects_pred"]
ref = scores["objects_ref"].str.split(";")  # "boat;water" -> ["boat", "water"]

# Turn each list into a row of 0s and 1s, one column per object name.
encoder = MultiLabelBinarizer()
encoder.fit(list(pred) + list(ref))
y_pred = encoder.transform(pred)
y_ref = encoder.transform(ref)

# F1 per image, averaged over all images.
f1 = f1_score(y_ref, y_pred, average="samples")
print("Object F1:", round(f1, 2))

**Text** is scored with the character error rate (CER): how many characters must be changed, per character of the label? Lower is better.

In [ ]:
# Keep only the images whose label contains text.
with_text = scores[scores["text_in_image_ref"].notna()]

# Label text and model text. No text from the model counts as empty text.
reference = list(with_text["text_in_image_ref"])
prediction = list(with_text["text_in_image_pred"].fillna(""))
print("Label:", reference)
print("Model:", prediction)

# Character error rate.
cer = jiwer.cer(reference, prediction)
print("CER:", round(cer, 2))

**Check yourself:** Why can a perfectly valid JSON answer still be wrong? Compare the two object lists of a few images.

In [ ]:
# Model and label objects side by side.
scores[["file", "objects_pred", "objects_ref"]]

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')